# Cursos de Series de tiempo con *Machine Learning*
## Modulo. Modelo Chronos Amazon
                        Elaborado por: Naren Castellon

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np
from chronos import ChronosPipeline

In [ ]:
df = pd.read_csv('./ferreteria.csv')
df = df.drop('Unnamed: 0', axis = 1)
df

In [ ]:
df.info()

In [ ]:
df['ds'] = pd.to_datetime(df["ds"])

In [ ]:
skus = [
    'Martillo 16oz', 'Destornillador Phillips', 'Taladro Inalámbrico', 'Sierra Circular', 'Pintura Blanca 1Gal',
    'Cemento 50kg', 'Tornillos 2"', 'Clavos 3"', 'Cinta Métrica 5m', 'Nivel 1m',
    'Llave Ajustable', 'Brochas 2"', 'Lija Grano 120', 'Pegamento PVC', 'Tubos PVC 1/2"'
]

In [ ]:
# --- 1. PREPARACIÓN DE DATOS HISTÓRICOS ---

# Convertimos los datos de demanda de cada SKU en una lista de tensores
# Chronos procesa cada serie de forma independiente en su modo base
series_data = []
for sku in skus:
    # Filtramos la demanda histórica por SKU y la convertimos a float32 para el modelo
    vendas_historicas = df[df['sku_id'] == sku]['demand_units'].values.astype(np.float32)
    series_data.append(torch.tensor(vendas_historicas))

print(f"✅ Datos preparados: {len(series_data)} SKUs listos para procesar.")

In [ ]:
# 2. Cargar el modelo optimizado para tu chip M1 Pro
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-mini", # 'tiny' es ultra rápido en Mac
    device_map="mps",         # Uso de GPU de Apple
    torch_dtype=torch.bfloat16,
)

In [ ]:
# --- 3. GENERACIÓN DEL PRONÓSTICO 2026 ---

# prediction_length = 52 (una predicción por cada lunes del 2026)
# num_samples = 20 (genera 20 trayectorias posibles para calcular la incertidumbre)
forecast = pipeline.predict(
    series_data, 
    prediction_length=52, 
    num_samples=200
)

In [ ]:
# --- 4. CREACIÓN DEL DATAFRAME DE RESULTADOS ---

# Definimos el rango de fechas de los lunes de 2026
dates_2026 = pd.date_range('2026-01-05', periods=52, freq='W-MON')
all_forecasts = []

for i, sku in enumerate(skus):
    # Extraemos los cuantiles: 
    # 0.1 (P10 - Escenario pesimista), 0.5 (Mediana - Probable), 0.9 (P90 - Optimista/Seguridad)
    quantiles = forecast[i].quantile(torch.tensor([0.1, 0.5, 0.9]), dim=0)
    
    sku_forecast_df = pd.DataFrame({
        'ds': dates_2026,
        'sku_id': sku,
        'demand_p10': quantiles[0].numpy(),
        'demand_median': quantiles[1].numpy(),
        'demand_p90': quantiles[2].numpy()
    })
    all_forecasts.append(sku_forecast_df)

# Unimos todo en un solo DataFrame final
df_final_2026 = pd.concat(all_forecasts, ignore_index=True)

In [ ]:
# --- 5. VISUALIZACIÓN DE RESULTADOS ---

def plot_sku_forecast(sku_name):
    plt.figure(figsize=(18, 6))
    
    # Datos históricos (último año para contexto)
    hist = df[df['sku_id'] == sku_name].tail(52)
    plt.plot(hist['ds'], hist['demand_units'], label='Historial 2025', color='black', alpha=0.6)
    
    # Pronóstico 2026
    pred = df_final_2026[df_final_2026['sku_id'] == sku_name]
    plt.plot(pred['ds'], pred['demand_median'], label='Pronóstico Mediana 2026', color='blue', lw=2)
    
    # Área de incertidumbre (P10 - P90)
    plt.fill_between(pred['ds'], pred['demand_p10'], pred['demand_p90'], 
                     color='blue', alpha=0.2, label='Intervalo de Confianza (80%)')
    
    plt.title(f"Pronóstico de Demanda Anual 2026: {sku_name}", fontsize=15)
    plt.ylabel("Unidades")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

In [ ]:
# Graficamos el primer SKU como ejemplo
plot_sku_forecast(skus[9])